!pip install pysindy

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from pysindy.feature_library import PolynomialLibrary, GeneralizedLibrary
from pysindy import SINDy
import pysindy as ps

In [2]:
#Define the system of ODEs
def system(t, variables, alpha):
    x, y, z = variables
    dxdt = alpha - x**2
    dydt = -y
    dzdt=0
    return [dxdt, dydt,dzdt]

In [6]:
# Initial conditions
initial_conditions = [0.5, 0.5,0]  # Initial values for x and y
alpha = 1  # Parameter alpha
threshold = 0.5

# Time span
t_span = (0, 10)
t_eval = np.arange(0, 10, 0.002)

# Solve the system
sol = solve_ivp(system, t_span, initial_conditions, t_eval=t_eval, args=(alpha,), dense_output=True)

# Extract the solution at the evaluated time points
x = sol.y.T

# Define the parameter and feature libraries
parameter_library = PolynomialLibrary(degree=1)
feature_library = PolynomialLibrary(degree=2)

# Create the parameterized library
parameterized_library = ps.ParameterizedLibrary(
    parameter_library=parameter_library,
    feature_library=feature_library,
    num_parameters=1,  # Number of parameters (alpha)
    num_features=2  # Number of features (x, y)
)

# SINDy modeling with the parameterized feature library
optimizer = ps.STLSQ(threshold=threshold)
model = ps.SINDy(
    optimizer=optimizer,
    feature_library=parameterized_library,
    feature_names=['x', 'y','z']
)

# Fit the SINDy model
model.fit(x, t=t_eval[1]-t_eval[0], u=alpha)
model.print()

(x)' = 1.002 1 1 + -1.002 1 x^2
(y)' = -1.000 1 y
(z)' = 0.000


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pysindy/optimizers/stlsq.py:201: UserWarning: Sparsity parameter is too big (0.5) and eliminated all coefficients
  warnings.warn(
